<a href="https://colab.research.google.com/github/sunitharamu-1983/GENAI-Learning-Journey/blob/main/Test_Search_Engine.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [25]:
#Step 0 - Mount the Google drive for Content
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [33]:
import os

#Step 1 - Read contents from TEST.txt file and load them to list

file_path = '/content/drive/MyDrive/GENAI 2026/Test.txt'

if not os.path.exists(file_path):
    print(f"Error: The file '{file_path}' does not exist. Please check the path and file name.")
else:
    with open(file_path, 'r') as f:
        contents = f.read()

        # If the file contains plain text with items on new lines:
        # Split the text by newline and remove any empty strings from the list
        results_list = [item.strip() for item in contents.split('\n') if item.strip()]

#Step 2 - List of contents from .TXT file loaded to a dictonary

documents = {"S"+ str(i): value for i, value in enumerate(results_list)}

print("\nStep 1 - Load to Dictinoary named Documents:")
for key, value in documents.items():
    print(f"{key}: {value}")

#Step 3 - Tokenization

print("\nStep 2 - Tokenization")
for doc_id,text in documents.items():
  text_list = text.lower().split()
  print(doc_id,":",len(text_list),"Tokens","-",text_list)

#Step 4 - STOPWORDS using STEMMING

import nltk
from nltk.corpus import stopwords
nltk.download('stopwords')
STOPWORDS = stopwords.words('english')

text_list_stopwords_removal={}
for doc_id in documents:
  text_list = documents[doc_id].lower().split()
  text_list_stopwords_removal[doc_id] = [word for word in text_list if word not in STOPWORDS]

print("\nStep 3 - STOPWORDS")
print(text_list_stopwords_removal)

#Step 5 - Stemming and Lemmatization
#Lemmatization takes longer to execute than stemming. Stemming is faster and so used widely

from nltk.stem import PorterStemmer
from nltk.stem import WordNetLemmatizer
nltk.download('wordnet')

processed = {}
for doc_id,text in documents.items():
  text_list = text.lower().split()
  processed[doc_id]=[WordNetLemmatizer().lemmatize(word) for word in text_list if word not in STOPWORDS]

for key, value in processed.items():
    print(f"{key}: {value}")

#Step 6 - Inverted Index

'''
The primary reason we use an inverted_index is for efficient retrieval of candidate documents, not for ranking or re-ranking itself.
Think of it as a dictionary where you can quickly look up a word and immediately get a list of all documents that contain that word.
This is much faster than scanning every document in your collection for every query.

Here's how it fits into the search process:
Retrieval (using Inverted Index): When you search for a query like "best pizza recipe",
the inverted index quickly identifies all documents that contain any of these terms.
Ranking (using TF-IDF, etc.): Once you have this smaller set of candidate documents,
you then apply ranking algorithms (like TF-IDF, which we just covered) to score and order these candidates based on their relevance to the query.
So, the inverted index is a crucial first step for speed, allowing the ranking algorithms to work on a manageable subset of documents.
'''

from collections import defaultdict
inverted_index = defaultdict(set)

unique_words = set()
for doc_id,text in processed.items():
  unique_words.update(text)

for doc_id, text in processed.items():
  for word in text:
    inverted_index[word].add(doc_id)

print("\nPrinting inverted_index in a readable format:")
for key, value_set in inverted_index.items():
    print(f"{key}: {list(value_set)}") # Convert set to list for a more standard print format

print("\nPrinting from unique_words:")
for word in unique_words:
  docs = sorted(inverted_index[word])
  print(word, " : ", docs)

#Step 7 - Term Frequency (TF)
# TF = count of word in this doc / total words in this doc

def tf(term, doc_terms):
    """
    term      : the word we're scoring (already stemmed/processed)
    doc_terms : list of processed tokens in the document
    """
    count      = doc_terms.count(term)  # 👇 how many times does 'term' appear?
    total      = len(doc_terms)         # 👇 total words in this document
    return count / total                # 👇 proportion of this doc that is 'term'

# --- Demonstration ---
print("TF scores for the word 'pizza' across all documents:")
print()
print(f"  {'Doc':<5} {'Tokens':<45} {'pizza count':>12} {'total words':>12} {'TF':>8}")
print("  " + "-" * 85)

query = "best"
for key, value_set in inverted_index.items():
  if query == key and len(value_set) > 0:
    print(f"{key}: {list(value_set)}")
    query_list=list(value_set)

for id in query_list:
  for doc_id, terms in processed.items():
    if doc_id == id:
      score = tf(query, terms)
      count = terms.count(query)
      marker = " ←" if score > 0 else ""
      print(f"  {doc_id:<5} {str(terms):<45} {count:>12} {len(terms):>12} {score:>8.4f}{marker}")



Step 1 - Load to Dictinoary named Documents:
S0: The best homemade pizza recipe for beginners
S1: How to make the best pizza dough from scratch recipe
S2: Top 10 tips for baking pizza at home for beginners
S3: Easy pasta recipes and Italian cooking techniques
S4: A guide to building your own outdoor pizza oven in summer
S5: The history of pizza from Naples to New York for foodies
S6: Healthy meal prep ideas and cooking tips for temperature control
S7: Best wood-fired pizza restaurants in the best cities
S8: How to choose the right flour for homemade bread and pizza
S9: Italian cheese varieties and their uses in traditional cooking

Step 2 - Tokenization
S0 : 7 Tokens - ['the', 'best', 'homemade', 'pizza', 'recipe', 'for', 'beginners']
S1 : 10 Tokens - ['how', 'to', 'make', 'the', 'best', 'pizza', 'dough', 'from', 'scratch', 'recipe']
S2 : 10 Tokens - ['top', '10', 'tips', 'for', 'baking', 'pizza', 'at', 'home', 'for', 'beginners']
S3 : 7 Tokens - ['easy', 'pasta', 'recipes', 'and', 'i

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


# **Retrieval (using Inverted Index)**

*When you search for a query like "best pizza recipe", the inverted index quickly identifies all documents that contain any of these terms. - How does unique_words fit here?*

That's a great question for clarification! While unique_words is crucial for building the inverted_index and understanding your corpus's vocabulary, it doesn't directly participate in the retrieval step when you search for a query.

**Here's how it fits:**

**Building the Inverted Index:** unique_words collects all the distinct terms that will become the keys in your inverted_index. It ensures that every unique word from your documents is accounted for and has an entry in the inverted index structure.

**During Query Retrieval**: When you have a query like "best pizza recipe", the process is:

**Process the Query:** The query itself is tokenized, stopwords removed, and lemmatized (e.g., to ['best', 'pizza', 'recipe']).

**Direct Lookup in inverted_index:** For each processed term in your query (e.g., 'best'), the system goes directly to the inverted_index and retrieves the set of doc_ids associated with that term. It does not iterate through unique_words during this lookup.

***So, unique_words is like the blueprint for the vocabulary that the inverted_index is built upon. The inverted_index is the actual lookup table that allows for fast retrieval using the terms defined in that vocabulary.***

# **How Inverted Index Helps with Term Frequency (TF)**

The `inverted_index` and Term Frequency (TF) work together in a search system, but they serve different primary purposes:

1.  **Inverted Index: For Efficient Retrieval**
    *   Its main goal is to **quickly find all documents that contain a given search term.** When you query for a word (or multiple words), the inverted index directly points you to the list of documents where that word appears. It acts as a pre-computed lookup table.
    *   **How it helps with TF:** By providing a rapid list of *candidate documents* that contain your query terms, the inverted index makes it feasible to calculate TF (and other scoring metrics like IDF) only on this smaller, relevant subset of documents, rather than scanning every document in your entire collection. This significantly speeds up the search process.

2.  **Term Frequency (TF): For Ranking and Scoring**
    *   Term Frequency measures **how often a term appears in a specific document.** A higher TF value for a term in a document generally suggests that the document is more relevant to that term.
    *   TF is a crucial component of scoring algorithms, most notably **TF-IDF (Term Frequency-Inverse Document Frequency)**, which is used to determine the relevance of documents to a query and to rank them.

**How they work together:**

*   When a user submits a query (e.g., "best pizza recipe"), the system first uses the **inverted index** to retrieve all documents that contain *any* of the query terms (`best`, `pizza`, `recipe`). This gives you a preliminary set of potentially relevant documents.
*   Once you have this set of candidate documents, the system then calculates the **Term Frequency (TF)** (and often IDF) for each query term within *these specific candidate documents*. These TF-IDF scores are then combined to produce a final relevance score for each document, which is used to rank them and present the most relevant ones to the user.

In essence, the `inverted_index` is the **gatekeeper** that quickly narrows down the search space, and TF (as part of TF-IDF) is the **judge** that ranks the documents within that narrowed space based on their actual content.

In [40]:
import math

def idf(term, num_total_docs, inverted_index):
    """
    term             : the word we're scoring
    num_total_docs   : total number of documents in the collection
    inverted_index   : the inverted index containing term-to-doc_ids mapping
    """
    df = len(inverted_index.get(term, set()))  # Efficiently get df from inverted_index
    if df == 0:
        return 0
    return math.log(num_total_docs / df)

# --- See IDF for various words ---
words_to_check = ['the','pizza', 'best', 'recipe']
num_total_docs = len(processed)

print("\n")
print(f"IDF scores across {num_total_docs} documents:")
print()
print(f"  {'Word':<15} {'Docs with it':>13} {'IDF score':>10}  Meaning")
print("  " + "-" * 65)

for word in words_to_check:
    df_count = len(inverted_index.get(word, set())) # Efficiently get df from inverted_index
    score = idf(word, num_total_docs, inverted_index) # Pass num_total_docs and inverted_index

    if df_count > 0:
        meaning = "very common → low signal" if score < 0.5 else (
                  "common"                   if score < 1.0 else (
                  "moderately rare"          if score < 2.0 else
                  "RARE → high signal"))
        print(f"  {word:<15} {df_count:>5}/{num_total_docs:<8} {score:>10.4f}  {meaning}")
    else:
        print(f"  {word:<15} {df_count:>5}/{num_total_docs:<8} {0.0000:>10.4f}  Not found")



IDF scores across 10 documents:

  Word             Docs with it  IDF score  Meaning
  -----------------------------------------------------------------
pizza: ['S8', 'S0', 'S5', 'S1', 'S7', 'S2', 'S4']
  pizza               7/10           0.3567  very common → low signal
  pizza               7/10           0.3567  very common → low signal
  pizza               7/10           0.3567  very common → low signal
  pizza               7/10           0.3567  very common → low signal
  pizza               7/10           0.3567  very common → low signal
  pizza               7/10           0.3567  very common → low signal
  pizza               7/10           0.3567  very common → low signal
best: ['S1', 'S0', 'S7']
  best                3/10           1.2040  moderately rare
  best                3/10           1.2040  moderately rare
  best                3/10           1.2040  moderately rare
recipe: ['S1', 'S3', 'S0']
  recipe              3/10           1.2040  moderately rare
  recipe 